In [1]:
# =====================================================================
# Hierarchical Transformer-Enhanced Network Intrusion Detection System
# 完整训练和正确打包版本
# =====================================================================

# Cell 1: 环境准备和导入库
import subprocess
import sys

# 安装必要的包
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "scikit-learn", "imbalanced-learn"], 
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import numpy as np
import pandas as pd
import os
import gc
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from collections import Counter

from tqdm import tqdm
import warnings
import math
import json
import pickle
from datetime import datetime
warnings.filterwarnings('ignore')

# 设置随机种子
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"环境初始化完成，使用设备: {device}")

# =====================================================================
# Cell 2: 保存模型架构文件
# =====================================================================

model_architecture_code = '''"""
Hierarchical Transformer-Enhanced Network Intrusion Detection System
Model Architecture Definition
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiScaleAttention(nn.Module):
    """多尺度注意力机制：结合类别、时间、空间注意力"""
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.num_classes = num_classes
        self.input_dim = input_dim
        
        # 类别特定注意力
        self.class_attention = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, input_dim // 4),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(input_dim // 4, input_dim),
                nn.Sigmoid()
            ) for _ in range(num_classes)
        ])
        
        # 时间序列注意力
        self.temporal_attention = nn.MultiheadAttention(
            input_dim, num_heads=4, dropout=0.1, batch_first=True
        )
        
        # 空间特征注意力
        self.spatial_attention = nn.Sequential(
            nn.Linear(input_dim, input_dim // 8),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(input_dim // 8, input_dim),
            nn.Sigmoid()
        )
        
        # 注意力融合权重
        self.fusion_weights = nn.Parameter(torch.ones(3) / 3)
    
    def forward(self, x):
        batch_size = x.size(0)
        
        # 1. 类别特定注意力
        class_features = []
        for i in range(self.num_classes):
            attention_weights = self.class_attention[i](x)
            attended_features = attention_weights * x
            class_features.append(attended_features)
        class_attended = torch.stack(class_features, dim=1)
        
        # 2. 时间序列注意力
        x_temporal = x.unsqueeze(1)
        temporal_attended, _ = self.temporal_attention(x_temporal, x_temporal, x_temporal)
        temporal_attended = temporal_attended.squeeze(1)
        
        # 3. 空间特征注意力
        spatial_weights = self.spatial_attention(x)
        spatial_attended = spatial_weights * x
        
        # 4. 多尺度融合
        weights = F.softmax(self.fusion_weights, dim=0)
        class_attended_mean = class_attended.mean(dim=1)
        
        fused_features = (weights[0] * class_attended_mean + 
                         weights[1] * temporal_attended + 
                         weights[2] * spatial_attended)
        
        return class_attended, fused_features

class TransformerEnhancedEnsembleModel(nn.Module):
    """Transformer增强的集成模型：支持层次化检测架构"""
    def __init__(self, input_dim, num_classes, dropout_rate=0.3, use_pretrained=False):
        super().__init__()
        self.num_classes = num_classes
        self.use_pretrained = use_pretrained
        
        # 共享编码器
        self.shared_encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate / 2)
        )
        
        # Transformer编码器
        self.feature_dim = 256
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.feature_dim,
            nhead=8,
            dim_feedforward=512,
            dropout=0.1,
            batch_first=True,
            activation='gelu'
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        # 位置编码
        self.pos_encoding = nn.Parameter(torch.randn(1, 1, self.feature_dim) * 0.1)
        
        # 多尺度注意力机制
        self.multi_scale_attention = MultiScaleAttention(256, num_classes)
        
        # 类别特定分类头
        self.class_specific_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(256, 128),
                nn.BatchNorm1d(128),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, 1)
            ) for _ in range(num_classes)
        ])
        
        # 全局分类器
        self.global_classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, num_classes)
        )
        
        # 自适应融合网络
        self.fusion_network = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
            nn.Softmax(dim=-1)
        )
        
        # 特征提取器
        self.feature_extractor = nn.Identity()
        
        # 不确定性估计头
        self.uncertainty_head = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x, return_features=False, return_uncertainty=False):
        # 共享特征提取
        shared_features = self.shared_encoder(x)
        
        if return_features:
            return shared_features
        
        # Transformer处理
        transformer_input = shared_features.unsqueeze(1) + self.pos_encoding
        transformer_features = self.transformer_encoder(transformer_input)
        transformer_features = transformer_features.squeeze(1)
        
        # 残差连接 + Layer Normalization
        enhanced_features = F.layer_norm(
            shared_features + transformer_features, 
            normalized_shape=[self.feature_dim]
        )
        
        # 多尺度注意力处理
        class_attended_features, fused_attention_features = self.multi_scale_attention(enhanced_features)
        
        # 类别特定输出
        class_specific_outputs = []
        for i in range(self.num_classes):
            output = self.class_specific_heads[i](class_attended_features[:, i, :])
            class_specific_outputs.append(output)
        class_specific_logits = torch.cat(class_specific_outputs, dim=1)
        
        # 全局输出
        global_logits = self.global_classifier(fused_attention_features)
        
        # 自适应融合
        fusion_weights = self.fusion_network(enhanced_features)
        final_logits = (fusion_weights[:, 0:1] * class_specific_logits + 
                       fusion_weights[:, 1:2] * global_logits)
        
        # 不确定性估计
        if return_uncertainty:
            uncertainty = self.uncertainty_head(enhanced_features)
            return final_logits, uncertainty
        
        return final_logits
    
    def load_pretrained_encoder(self, pretrained_model_path):
        """加载预训练的编码器权重"""
        import os
        if os.path.exists(pretrained_model_path):
            pretrained_state = torch.load(pretrained_model_path, map_location='cpu')
            encoder_state = {}
            for key, value in pretrained_state.items():
                if key.startswith('shared_encoder'):
                    encoder_state[key] = value
            
            self.load_state_dict(encoder_state, strict=False)
'''

# 保存model_architecture.py
with open('/kaggle/working/model_architecture.py', 'w') as f:
    f.write(model_architecture_code)
print("✅ model_architecture.py 已保存")

# 导入模型架构
from model_architecture import TransformerEnhancedEnsembleModel, MultiScaleAttention

# =====================================================================
# Cell 3: 改进的损失函数和训练工具
# =====================================================================

class UncertaintyAwareFocalLoss(nn.Module):
    """不确定性感知的自适应焦点损失"""
    def __init__(self, alpha=None, gamma=2.0, class_specific_gamma=None, uncertainty_weight=1.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.class_specific_gamma = class_specific_gamma or {}
        self.uncertainty_weight = uncertainty_weight
        
    def forward(self, inputs, targets, uncertainty=None):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        
        gamma_values = torch.full_like(targets, self.gamma, dtype=torch.float)
        for class_idx, gamma_val in self.class_specific_gamma.items():
            mask = (targets == class_idx)
            gamma_values[mask] = gamma_val
        
        focal_loss = (1 - pt) ** gamma_values * ce_loss
        
        if uncertainty is not None:
            uncertainty_weight = 1 + self.uncertainty_weight * uncertainty.squeeze()
            focal_loss = uncertainty_weight * focal_loss
        
        if self.alpha is not None:
            if self.alpha.device != inputs.device:
                self.alpha = self.alpha.to(inputs.device)
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss
        
        return focal_loss.mean()

def train_transformer_model(model, train_loader, val_loader, device, config):
    """改进的Transformer模型训练函数"""
    print(f"开始训练 {config['model_name']}")
    
    if config.get('use_pretrained', False):
        encoder_params = []
        transformer_params = []
        other_params = []
        
        for name, param in model.named_parameters():
            if 'shared_encoder' in name:
                encoder_params.append(param)
            elif 'transformer' in name or 'pos_encoding' in name:
                transformer_params.append(param)
            else:
                other_params.append(param)
        
        optimizer = optim.AdamW([
            {'params': encoder_params, 'lr': config['lr'] * 0.1},
            {'params': transformer_params, 'lr': config['lr'] * 0.5},
            {'params': other_params, 'lr': config['lr']}
        ], weight_decay=config.get('weight_decay', 1e-4))
    else:
        optimizer = optim.AdamW(model.parameters(), lr=config['lr'], 
                              weight_decay=config.get('weight_decay', 1e-4))
    
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=config['lr'] if not config.get('use_pretrained', False) else [config['lr']*0.1, config['lr']*0.5, config['lr']],
        epochs=config['num_epochs'],
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
        anneal_strategy='cos'
    )
    
    if 'class_weights' in config and config['class_weights'] is not None:
        alpha = torch.FloatTensor(config['class_weights']).to(device)
    else:
        alpha = None
    
    class_specific_gamma = {}
    if 'minority_classes' in config:
        for class_idx in config['minority_classes']:
            class_specific_gamma[class_idx] = 3.0
    
    criterion = UncertaintyAwareFocalLoss(
        alpha=alpha, 
        gamma=2.0,
        class_specific_gamma=class_specific_gamma,
        uncertainty_weight=0.5
    )
    
    best_val_f1 = 0.0
    patience_counter = 0
    best_model_path = f"/kaggle/working/best_{config['model_name']}.pth"
    
    for epoch in range(config['num_epochs']):
        model.train()
        total_loss = 0
        train_preds, train_labels = [], []
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config['num_epochs']}")
        for batch_idx, (inputs, labels) in enumerate(pbar):
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs, uncertainty = model(inputs, return_uncertainty=True)
            loss = criterion(outputs, labels, uncertainty)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
            
            _, predicted = torch.max(outputs, 1)
            train_preds.extend(predicted.cpu().numpy())
            train_labels.extend(labels.cpu().numpy())
            
            pbar.set_postfix({'Loss': f'{loss.item():.4f}'})
        
        model.eval()
        val_loss = 0
        val_preds, val_labels = [], []
        val_uncertainties = []
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs, uncertainty = model(inputs, return_uncertainty=True)
                loss = criterion(outputs, labels, uncertainty)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs, 1)
                val_preds.extend(predicted.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
                val_uncertainties.extend(uncertainty.cpu().numpy())
        
        train_f1 = f1_score(train_labels, train_preds, average='macro')
        val_f1 = f1_score(val_labels, val_preds, average='macro')
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}: 训练F1: {train_f1:.4f} | 验证F1: {val_f1:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_counter = 0
            torch.save(model.state_dict(), best_model_path)
        else:
            patience_counter += 1
            if patience_counter >= config.get('patience', 7):
                print(f"早停触发，最佳F1: {best_val_f1:.4f}")
                break
    
    return best_model_path, best_val_f1

# =====================================================================
# Cell 4: 数据加载和预处理
# =====================================================================

def load_and_preprocess_data():
    """加载和预处理CIC-IDS2017数据"""
    print("数据加载和预处理中...")
    
    data_path = '/kaggle/input/cicids2017'
    
    parquet_files = [os.path.join(data_path, f) for f in os.listdir(data_path) 
                     if f.endswith('.parquet')]
    
    df_list = []
    for file in tqdm(parquet_files, desc="加载数据文件", leave=False):
        df_temp = pd.read_parquet(file)
        df_list.append(df_temp)
    
    df = pd.concat(df_list, ignore_index=True)
    del df_list
    gc.collect()
    
    print(f"数据形状: {df.shape}")
    
    df.rename(columns={col: col.strip() for col in df.columns}, inplace=True)
    label_column = 'Label'
    
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)
    
    numeric_columns = df.select_dtypes(include=np.number).columns
    numeric_columns = [col for col in numeric_columns if col != label_column]
    
    for col in tqdm(numeric_columns, desc="数据清理", leave=False):
        q99, q01 = df[col].quantile(0.99), df[col].quantile(0.01)
        df[col] = df[col].clip(lower=q01, upper=q99)
    
    rows_before = df.shape[0]
    df.drop_duplicates(inplace=True)
    if rows_before != df.shape[0]:
        print(f"移除了 {rows_before - df.shape[0]:,} 条重复记录")
    
    df[label_column] = df[label_column].astype(str).str.replace(
        r'[^a-zA-Z0-9\s-]', '', regex=True
    ).str.replace(r'\s+', ' ', regex=True).str.strip()
    
    df['Binary_Label'] = df[label_column].apply(
        lambda x: 'Benign' if x == 'Benign' else 'Malicious'
    )
    
    multi_class_mapping = {
        'DoS Hulk': 'DoS',
        'DoS GoldenEye': 'DoS', 
        'DoS slowloris': 'DoS',
        'DoS Slowhttptest': 'DoS',
        'FTP-Patator': 'Brute_Force',
        'SSH-Patator': 'Brute_Force',
        'Web Attack Brute Force': 'Web_Attack',
        'Web Attack XSS': 'Web_Attack',
        'Web Attack Sql Injection': 'Web_Attack',
        'PortScan': 'PortScan',
        'Bot': 'Bot',
        'Infiltration': 'Rare_Attacks',
        'Heartbleed': 'Rare_Attacks'
    }
    
    # 添加DDoS映射
    df[label_column] = df[label_column].apply(
        lambda x: 'DDoS' if 'ddos' in x.lower() else x
    )
    multi_class_mapping['DDoS'] = 'DDoS'
    
    df['Multi_Label'] = df[label_column].replace(multi_class_mapping)
    df = df[~df['Multi_Label'].isin(['Rare_Attacks'])]
    
    print("标签分布:")
    print(df['Multi_Label'].value_counts())
    
    feature_columns = [col for col in df.columns 
                      if col not in [label_column, 'Binary_Label', 'Multi_Label']]
    X = df[feature_columns].copy()
    
    le_binary = LabelEncoder()
    y_binary = le_binary.fit_transform(df['Binary_Label'])
    
    le_multi = LabelEncoder()
    y_multi = le_multi.fit_transform(df['Multi_Label'])
    
    print(f"特征维度: {X.shape[1]}")
    print(f"多分类类别: {list(le_multi.classes_)}")
    
    return X, y_binary, y_multi, le_binary, le_multi

X, y_binary, y_multi, le_binary, le_multi = load_and_preprocess_data()

# =====================================================================
# Cell 5: 第一阶段 - 训练二分类预训练模型
# =====================================================================

print("\n=== 第一阶段：训练Transformer增强的二分类预训练模型 ===")

X_train, X_test, y_train_binary, y_test_binary = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)
X_train, X_val, y_train_binary, y_val_binary = train_test_split(
    X_train, y_train_binary, test_size=0.2, random_state=42, stratify=y_train_binary
)

# 关键：使用scaler_binary进行标准化
scaler_binary = StandardScaler()
X_train_scaled = scaler_binary.fit_transform(X_train)
X_val_scaled = scaler_binary.transform(X_val)
X_test_scaled = scaler_binary.transform(X_test)

print(f"训练集大小: {len(X_train_scaled):,}")

class_weights_binary = compute_class_weight(
    'balanced', 
    classes=np.unique(y_train_binary), 
    y=y_train_binary
)

train_loader_binary = DataLoader(
    TensorDataset(torch.from_numpy(X_train_scaled).float(), 
                 torch.from_numpy(y_train_binary).long()),
    batch_size=1024, shuffle=True, num_workers=2
)

val_loader_binary = DataLoader(
    TensorDataset(torch.from_numpy(X_val_scaled).float(), 
                 torch.from_numpy(y_val_binary).long()),
    batch_size=1024, num_workers=2
)

binary_model = TransformerEnhancedEnsembleModel(
    input_dim=X.shape[1], 
    num_classes=2, 
    dropout_rate=0.3
).to(device)

print(f"模型参数量: {sum(p.numel() for p in binary_model.parameters()):,}")

binary_config = {
    'model_name': 'TransformerBinary_Pretrain',
    'num_epochs': 20,
    'lr': 0.001,
    'weight_decay': 1e-4,
    'patience': 5,
    'class_weights': class_weights_binary,
    'minority_classes': [1],
    'use_pretrained': False
}

best_binary_path, best_binary_f1 = train_transformer_model(
    binary_model, train_loader_binary, val_loader_binary, device, binary_config
)

print(f"二分类预训练完成，最佳F1: {best_binary_f1:.4f}")

del train_loader_binary, val_loader_binary
gc.collect()

# =====================================================================
# Cell 6: 第二阶段 - 准备多分类数据和分层采样
# =====================================================================

print("\n=== 第二阶段：准备多分类数据和分层采样 ===")

malicious_indices = (y_binary == 1)
X_malicious = X[malicious_indices].copy()
y_malicious_original = y_multi[malicious_indices].copy()

print(f"恶意流量样本数: {len(X_malicious):,}")

unique_labels = np.unique(y_malicious_original)
label_mapping = {old_label: new_label for new_label, old_label in enumerate(unique_labels)}
reverse_mapping = {new_label: old_label for old_label, new_label in label_mapping.items()}

y_malicious = np.array([label_mapping[label] for label in y_malicious_original])

le_multi_subset = LabelEncoder()
class_names_subset = [le_multi.classes_[reverse_mapping[i]] for i in range(len(unique_labels))]
le_multi_subset.classes_ = np.array(class_names_subset)

print("恶意流量类别分布:")
multi_class_counts = Counter(y_malicious)
for class_idx, count in sorted(multi_class_counts.items()):
    class_name = class_names_subset[class_idx]
    percentage = count / len(y_malicious) * 100
    print(f"  {class_name}: {count:,} 样本 ({percentage:.2f}%)")

tier1_classes = []
tier2_classes = []
total_malicious = len(y_malicious)

for class_idx, count in multi_class_counts.items():
    ratio = count / total_malicious
    if ratio < 0.05:
        tier1_classes.append(class_idx)
    elif ratio < 0.2:
        tier2_classes.append(class_idx)

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_malicious, y_malicious, test_size=0.2, random_state=42, stratify=y_malicious
)
X_train_m, X_val_m, y_train_m, y_val_m = train_test_split(
    X_train_m, y_train_m, test_size=0.2, random_state=42, stratify=y_train_m
)

print(f"多分类训练集: {len(X_train_m):,}")

max_count = max(Counter(y_train_m).values())
sampling_ratios = {}

for class_idx in range(len(class_names_subset)):
    current_count = sum(y_train_m == class_idx)
    if class_idx in tier1_classes:
        target_count = max(current_count, int(max_count * 0.4))
    elif class_idx in tier2_classes:
        target_count = max(current_count, int(max_count * 0.25))
    else:
        target_count = current_count
    
    sampling_ratios[class_idx] = target_count

try:
    smote = SMOTE(
        sampling_strategy=sampling_ratios,
        random_state=42,
        k_neighbors=min(5, min(Counter(y_train_m).values()) - 1)
    )
    
    X_train_m_resampled, y_train_m_resampled = smote.fit_resample(X_train_m, y_train_m)
    print(f"SMOTE采样完成: {len(X_train_m):,} → {len(X_train_m_resampled):,} 样本")
        
except Exception as e:
    print(f"SMOTE采样失败: {str(e)}，使用原始数据")
    X_train_m_resampled, y_train_m_resampled = X_train_m, y_train_m

# =====================================================================
# Cell 7: 第三阶段 - 端到端微调多分类模型
# =====================================================================

print("\n=== 第三阶段：端到端微调多分类模型 ===")

# 注意：多分类使用独立的scaler
scaler_multi = StandardScaler()
X_train_m_scaled = scaler_multi.fit_transform(X_train_m_resampled)
X_val_m_scaled = scaler_multi.transform(X_val_m)
X_test_m_scaled = scaler_multi.transform(X_test_m)

unique_classes = np.arange(len(class_names_subset))
class_weights_multi = compute_class_weight(
    'balanced', 
    classes=unique_classes, 
    y=y_train_m_resampled
)

train_loader_multi = DataLoader(
    TensorDataset(torch.from_numpy(X_train_m_scaled).float(), 
                 torch.from_numpy(y_train_m_resampled).long()),
    batch_size=512, shuffle=True, num_workers=0
)

val_loader_multi = DataLoader(
    TensorDataset(torch.from_numpy(X_val_m_scaled).float(), 
                 torch.from_numpy(y_val_m).long()),
    batch_size=512, num_workers=0
)

test_loader_multi = DataLoader(
    TensorDataset(torch.from_numpy(X_test_m_scaled).float(), 
                 torch.from_numpy(y_test_m).long()),
    batch_size=512, num_workers=0
)

num_classes = len(class_names_subset)

multi_model = TransformerEnhancedEnsembleModel(
    input_dim=X.shape[1], 
    num_classes=num_classes,
    dropout_rate=0.4,
    use_pretrained=True
).to(device)

multi_model.load_pretrained_encoder(best_binary_path)

multi_config = {
    'model_name': 'TransformerMultiClass_FineTuned',
    'num_epochs': 25,
    'lr': 0.001,
    'weight_decay': 1e-4,
    'patience': 8,
    'class_weights': class_weights_multi,
    'minority_classes': tier1_classes + tier2_classes,
    'use_pretrained': True
}

best_multi_path, best_multi_f1 = train_transformer_model(
    multi_model, train_loader_multi, val_loader_multi, device, multi_config
)

print(f"多分类微调完成，最佳F1: {best_multi_f1:.4f}")

label_mapping_info = {
    'original_to_new': label_mapping,
    'new_to_original': reverse_mapping,
    'class_names_subset': class_names_subset,
    'tier1_classes': tier1_classes,
    'tier2_classes': tier2_classes
}

with open('/kaggle/working/label_mapping.pkl', 'wb') as f:
    pickle.dump(label_mapping_info, f)

# =====================================================================
# Cell 8: 模型评估
# =====================================================================

def evaluate_transformer_model(model, model_path, test_loader, device, 
                               class_names, model_name):
    """评估Transformer模型性能"""
    print(f"\n评估 {model_name}")
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    y_true, y_pred, y_scores, uncertainties = [], [], [], []
    
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc="模型评估", leave=False):
            inputs = inputs.to(device)
            outputs, uncertainty = model(inputs, return_uncertainty=True)
            
            probs = F.softmax(outputs, dim=1)
            y_scores.extend(probs.cpu().numpy())
            
            _, predicted = torch.max(outputs, 1)
            y_pred.extend(predicted.cpu().numpy())
            y_true.extend(labels.numpy())
            uncertainties.extend(uncertainty.cpu().numpy())
    
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    uncertainties = np.array(uncertainties)
    
    max_label = max(y_true.max(), y_pred.max())
    if max_label >= len(class_names):
        y_true = np.clip(y_true, 0, len(class_names)-1)
        y_pred = np.clip(y_pred, 0, len(class_names)-1)
    
    unique_labels = np.unique(np.concatenate([y_true, y_pred]))
    actual_class_names = [class_names[i] for i in unique_labels if i < len(class_names)]
    
    try:
        report = classification_report(
            y_true, y_pred, 
            labels=unique_labels,
            target_names=actual_class_names,
            digits=4, 
            zero_division=0
        )
        print(f"\n{model_name} 分类报告:")
        print(report)
    except Exception as e:
        print(f"生成分类报告时出错: {e}")
    
    try:
        macro_f1 = f1_score(y_true, y_pred, labels=unique_labels, average='macro')
        weighted_f1 = f1_score(y_true, y_pred, labels=unique_labels, average='weighted')
    except:
        macro_f1 = 0.0
        weighted_f1 = 0.0
    
    overall_accuracy = np.mean(y_true == y_pred)
    avg_uncertainty = np.mean(uncertainties)
    
    print(f"\n{model_name} 整体性能:")
    print(f"  准确率: {overall_accuracy:.4f}")
    print(f"  宏平均F1: {macro_f1:.4f}")
    print(f"  加权平均F1: {weighted_f1:.4f}")
    print(f"  平均不确定性: {avg_uncertainty:.4f}")
    
    return {
        'y_true': y_true,
        'y_pred': y_pred,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
        'accuracy': overall_accuracy,
        'avg_uncertainty': avg_uncertainty,
        'class_names': actual_class_names,
        'unique_labels': unique_labels
    }

print("\n=== 层次化Transformer增强模型评估 ===")

test_loader_binary = DataLoader(
    TensorDataset(torch.from_numpy(X_test_scaled).float(), 
                 torch.from_numpy(y_test_binary).long()),
    batch_size=1024, num_workers=0
)

binary_class_names = le_binary.classes_.tolist()
binary_results = evaluate_transformer_model(
    binary_model, best_binary_path, test_loader_binary, device,
    binary_class_names, "Transformer增强二分类模型"
)

multi_results = evaluate_transformer_model(
    multi_model, best_multi_path, test_loader_multi, device,
    class_names_subset, "Transformer增强多分类模型"
)

print("\n=== 层次化Transformer增强检测系统效果总结 ===")

print(f"\n二分类模型性能:")
print(f"   准确率: {binary_results['accuracy']:.4f}")
print(f"   宏平均F1: {binary_results['macro_f1']:.4f}")

print(f"\n多分类模型性能:")
print(f"   准确率: {multi_results['accuracy']:.4f}")
print(f"   宏平均F1: {multi_results['macro_f1']:.4f}")

# =====================================================================
# Cell 9: 正确的模型打包 - 修复scaler bug
# =====================================================================

print("\n开始模型打包...")

model_package_dir = "/kaggle/working/model_package"
os.makedirs(model_package_dir, exist_ok=True)

print("保存6个核心文件...")

# 1. model.pth
print("  生成 model.pth...")
model_state = {
    'binary_model_state': torch.load(best_binary_path, map_location='cpu'),
    'multi_model_state': torch.load(best_multi_path, map_location='cpu'),
    'model_architecture': {
        'input_dim': X.shape[1],
        'binary_classes': 2,
        'multi_classes': len(class_names_subset),
        'dropout_rate': 0.3
    }
}
torch.save(model_state, os.path.join(model_package_dir, "model.pth"))

# 2. scaler.pkl - 关键修复：使用scaler_binary而不是scaler_multi！
print("  生成 scaler.pkl (使用正确的scaler_binary)...")
with open(os.path.join(model_package_dir, "scaler.pkl"), 'wb') as f:
    pickle.dump(scaler_binary, f)  # 修复：使用二分类的scaler！

# 3. label_encoder.pkl
print("  生成 label_encoder.pkl...")
label_encoders = {
    'binary_encoder': le_binary,
    'multi_encoder': le_multi_subset,
    'binary_classes': le_binary.classes_.tolist(),
    'multi_classes': class_names_subset,
    'label_mapping': label_mapping_info
}
with open(os.path.join(model_package_dir, "label_encoder.pkl"), 'wb') as f:
    pickle.dump(label_encoders, f)

# 4. model_info.json
print("  生成 model_info.json...")
model_info = {
    'model_name': 'HierarchicalTransformerIDS',
    'model_version': '1.0.0',
    'created_date': datetime.now().isoformat(),
    'architecture': {
        'input_features': len(X.columns),
        'binary_classes': 2,
        'multi_classes': len(class_names_subset),
        'dropout_rate': 0.3
    },
    'classes': {
        'binary': le_binary.classes_.tolist(),
        'multi': class_names_subset,
        'tier1_critical': [class_names_subset[i] for i in tier1_classes],
        'tier2_minority': [class_names_subset[i] for i in tier2_classes]
    },
    'performance': {
        'binary_stage': {
            'accuracy': float(binary_results['accuracy']),
            'macro_f1': float(binary_results['macro_f1']),
            'weighted_f1': float(binary_results.get('weighted_f1', 0))
        },
        'multi_stage': {
            'accuracy': float(multi_results['accuracy']),
            'macro_f1': float(multi_results['macro_f1']),
            'weighted_f1': float(multi_results.get('weighted_f1', 0))
        }
    },
    'features': {
        'total_features': len(X.columns),
        'feature_names': X.columns.tolist(),
        'preprocessing': 'StandardScaler + Outlier Clipping'
    },
    'deployment': {
        'input_format': 'numpy array of shape (batch_size, n_features)',
        'requires_preprocessing': True,
        'batch_inference': True,
        'scaler_type': 'binary_scaler'  # 明确标记使用的是二分类scaler
    }
}
with open(os.path.join(model_package_dir, "model_info.json"), 'w') as f:
    json.dump(model_info, f, indent=2, default=str)

# 5. feature_selector.pkl
print("  生成 feature_selector.pkl...")
feature_selector = {
    'feature_columns': X.columns.tolist(),
    'selected_features': X.columns.tolist(),
    'feature_count': len(X.columns),
    'selection_method': 'all_features'
}
with open(os.path.join(model_package_dir, "feature_selector.pkl"), 'wb') as f:
    pickle.dump(feature_selector, f)

# 6. selected_features.json
print("  生成 selected_features.json...")
selected_features = {
    'features': X.columns.tolist(),
    'count': len(X.columns),
    'creation_date': datetime.now().isoformat()
}
with open(os.path.join(model_package_dir, "selected_features.json"), 'w') as f:
    json.dump(selected_features, f, indent=2)

print("\n验证生成的文件...")
required_files = ["model.pth", "scaler.pkl", "label_encoder.pkl", 
                  "model_info.json", "feature_selector.pkl", "selected_features.json"]

all_good = True
for file in required_files:
    if os.path.exists(os.path.join(model_package_dir, file)):
        print(f"  ✅ {file}")
    else:
        print(f"  ❌ {file} - 缺失")
        all_good = False

if all_good:
    total_size = sum(os.path.getsize(os.path.join(model_package_dir, f)) 
                    for f in os.listdir(model_package_dir)) / (1024*1024)
    print(f"\n✅ 模型包生成完成!")
    print(f"📍 位置: {model_package_dir}")
    print(f"📊 大小: {total_size:.1f}MB")
    print(f"📦 包含6个核心文件:")
    for i, file in enumerate(required_files, 1):
        print(f"   {i}. {file}")
    
    print(f"\n📋 文件说明:")
    print(f"   • model.pth: 打包的模型权重(二分类+多分类)")
    print(f"   • scaler.pkl: 数据标准化缩放器(使用正确的binary scaler)")
    print(f"   • label_encoder.pkl: 标签编码信息") 
    print(f"   • model_info.json: 模型架构和性能信息")
    print(f"   • feature_selector.pkl: 特征选择器")
    print(f"   • selected_features.json: 训练特征列表")
    
    print(f"\n🔧 关键修复:")
    print(f"   已修复scaler bug - 现在使用正确的scaler_binary")
else:
    print("\n❌ 模型包生成不完整，请检查错误信息")

print("\n模型打包完成!")
print("\n现在可以下载 /kaggle/working/model_package 文件夹")
print("以及 /kaggle/working/model_architecture.py 文件")

环境初始化完成，使用设备: cuda
✅ model_architecture.py 已保存
数据加载和预处理中...


数据形状: (2313810, 78)


移除了 84,674 条重复记录
标签分布:
Multi_Label
Benign         1892659
DoS             193730
DDoS            128014
Brute_Force       9150
Web_Attack        2143
PortScan          1956
Bot               1437
Name: count, dtype: int64
特征维度: 77
多分类类别: ['Benign', 'Bot', 'Brute_Force', 'DDoS', 'DoS', 'PortScan', 'Web_Attack']

=== 第一阶段：训练Transformer增强的二分类预训练模型 ===
训练集大小: 1,426,616
模型参数量: 1,722,730
开始训练 TransformerBinary_Pretrain


Epoch 5/20: 100%|██████████| 1394/1394 [00:26<00:00, 53.10it/s, Loss=0.0024]


Epoch 5: 训练F1: 0.9831 | 验证F1: 0.9829


...

Epoch 10: 训练F1: 0.9916 | 验证F1: 0.9297


Epoch 11/25: 100%|██████████| 790/790 [00:19<00:00, 40.68it/s, Loss=0.0009]


早停触发，最佳F1: 0.9687
多分类微调完成，最佳F1: 0.9687

=== 层次化Transformer增强模型评估 ===

评估 Transformer增强二分类模型



Transformer增强二分类模型 分类报告:
              precision    recall  f1-score   support

      Benign     0.9979    0.9960    0.9970    378532
   Malicious     0.9777    0.9883    0.9830     67286

    accuracy                         0.9948    445818
   macro avg     0.9878    0.9922    0.9900    445818
weighted avg     0.9949    0.9948    0.9948    445818


Transformer增强二分类模型 整体性能:
  准确率: 0.9948
  宏平均F1: 0.9900
  加权平均F1: 0.9948
  平均不确定性: 0.0001

评估 Transformer增强多分类模型



Transformer增强多分类模型 分类报告:
              precision    recall  f1-score   support

         Bot     0.9379    1.0000    0.9680       287
 Brute_Force     1.0000    0.9781    0.9890      1830
        DDoS     0.9999    0.9993    0.9996     25603
         DoS     0.9994    0.9979    0.9986     38746
    PortScan     0.9842    0.9540    0.9688       391
  Web_Attack     0.7992    0.9930    0.8857       429

    accuracy                         0.9976     67286
   macro avg     0.9534    0.9871    0.9683     67286
weighted avg     0.9979    0.9976    0.9977     67286


Transformer增强多分类模型 整体性能:
  准确率: 0.9976
  宏平均F1: 0.9683
  加权平均F1: 0.9977
  平均不确定性: 0.0027

=== 层次化Transformer增强检测系统效果总结 ===

二分类模型性能:
   准确率: 0.9948
   宏平均F1: 0.9900

多分类模型性能:
   准确率: 0.9976
   宏平均F1: 0.9683

开始模型打包...
保存6个核心文件...
  生成 model.pth...
  生成 scaler.pkl (使用正确的scaler_binary)...
  生成 label_encoder.pkl...
  生成 model_info.json...
  生成 feature_selector.pkl...
  生成 selected_features.json...

验证生成的文件...
  ✅ model.pth
  ✅ sc

In [2]:
# =====================================================================
# Cell 9.5: 保存分位数阈值 - 新增
# =====================================================================

import os
import pickle

# 重新加载必要的变量（如果需要）
model_package_dir = "/kaggle/working/model_package"

# 7. quantile_thresholds.pkl - 关键新增：保存训练时的分位数阈值
print("生成 quantile_thresholds.pkl (训练时的固定分位数阈值)...")
quantile_thresholds = {}

# 重新计算训练时使用的分位数阈值
numeric_columns = X.select_dtypes(include=np.number).columns
for col in numeric_columns:
    q01 = X[col].quantile(0.01)
    q99 = X[col].quantile(0.99)
    quantile_thresholds[col] = {
        'q01': float(q01),
        'q99': float(q99)
    }

with open(os.path.join(model_package_dir, "quantile_thresholds.pkl"), 'wb') as f:
    pickle.dump(quantile_thresholds, f)

print(f"✅ 保存了 {len(quantile_thresholds)} 个特征的分位数阈值 (1%-99%)")
print(f"📍 文件位置: {model_package_dir}/quantile_thresholds.pkl")

# 验证文件是否成功创建
if os.path.exists(os.path.join(model_package_dir, "quantile_thresholds.pkl")):
    file_size = os.path.getsize(os.path.join(model_package_dir, "quantile_thresholds.pkl")) / 1024
    print(f"📊 文件大小: {file_size:.2f} KB")
    print("🔧 现在预测时可以使用固定的分位数阈值，避免数据泄露")
else:
    print("❌ 文件创建失败")

生成 quantile_thresholds.pkl (训练时的固定分位数阈值)...
✅ 保存了 77 个特征的分位数阈值 (1%-99%)
📍 文件位置: /kaggle/working/model_package/quantile_thresholds.pkl
📊 文件大小: 3.36 KB
🔧 现在预测时可以使用固定的分位数阈值，避免数据泄露
